# Evaluating Reweighing on three binary classification datasets

Calders, T., Kamiran, F., & Pechenizkiy, M. (2009). *Building Classifiers with
Independency Constraints*, §3.2.

- [German Credit](https://doi.org/10.24432/C5NC77), as encoded in
  [caret](https://cran.r-project.org/package=caret): good credit risk is 1.
  Applicants aged 25 and below are group 0, and older applicants are reference
  group 1. Age is removed from the predictors.
- [Income survey](https://cran.r-project.org/package=kernlab) (`kernlab::income`):
  the two original income bins starting at $50,000 are mapped to 1.
  Recorded female is group 0 and male is reference group 1.
- [Titanic](https://cran.r-project.org/package=titanic) (`titanic_train`, the
  891 labeled passenger records): survival is 1. Recorded male is group 0
  and female is reference group 1, reflecting the direction of the historical
  survival gap. Use passenger class, age, siblings/spouses aboard,
  parents/children aboard, fare and embarkation port.

In [5]:
from pathlib import Path
import subprocess
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
TEST_SIZE = 0.20
NEIGHBORS = 5

def load_datasets():
    export = '''
    target <- commandArgs(trailingOnly=TRUE)[1]
    data("GermanCredit", package="caret")
    data("income", package="kernlab")
    write.csv(GermanCredit, file.path(target, "german.csv"), row.names=FALSE)
    write.csv(income, file.path(target, "income.csv"), row.names=FALSE)
    write.csv(titanic::titanic_train, file.path(target, "titanic.csv"), row.names=FALSE)
    '''
    with TemporaryDirectory() as directory:
        subprocess.run(
            ["Rscript", "--vanilla", "-e", export, directory],
            check=True, capture_output=True, text=True,
        )
        german, income, titanic = [
            pd.read_csv(Path(directory) / f"{name}.csv")
            for name in ("german", "income", "titanic")
        ]

    titanic_features = ["Pclass", "Age", "SibSp", "Parch", "Fare", "Embarked"]
    titanic["Pclass"] = titanic["Pclass"].astype(str)
    return {
        "German Credit": (
            german.drop(columns=["Age", "Class"]),
            german["Class"].eq("Good").to_numpy(dtype=int),
            german["Age"].gt(25).to_numpy(dtype=int),
        ),
        "Income survey": (
            income.drop(columns=["INCOME", "SEX"]),
            income["INCOME"].isin(["[50.000-75.000)", "[75.000-"]).to_numpy(dtype=int),
            income["SEX"].map({"F": 0, "M": 1}).to_numpy(dtype=int),
        ),
        "Titanic": (
            titanic[titanic_features],
            titanic["Survived"].to_numpy(dtype=int),
            titanic["Sex"].map({"male": 0, "female": 1}).to_numpy(dtype=int),
        ),
    }

datasets = load_datasets()

## Fairness metrics and full-data label audit

Let $p_0$ and $p_1$ be the favorable-label rates for group 0 and the
reference group. Use the signed definition $\mathrm{DP\_Diff}=p_0-p_1$
and $\mathrm{DIR}=p_0/p_1$ (undefined if $p_1=0$). Parity corresponds to 0
and 1, respectively. The paper uses the reverse sign for its dependency gap.

Consistency is $1-\operatorname{mean}_i |z_i - \operatorname{mean}_{j\in N_5(i)}z_j|$
for a binary label or prediction vector $z$. Neighborhoods are Euclidean in
the imputed/encoded feature space. Each row explicitly counts as one of its
five neighbors. Distances exclude the protected attribute and target.
A fixed reference set can contain tied distances, particularly for categorical
records. The result depends on feature representation and the tie order.
[Consistency definition](https://aif360.readthedocs.io/en/latest/modules/generated/aif360.sklearn.metrics.consistency_score.html)

Fit the full-data encoder **only** to describe the original labels. The
subsequent modeling encoder is fitted independently on the training set.

In [6]:
def make_encoder(features):
    numeric = features.select_dtypes(include="number").columns.tolist()
    categorical = features.columns.difference(numeric, sort=False).tolist()

    binary = binary_columns(features, numeric)
    return ColumnTransformer(
        [
            ("continuous", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
             [column for column in numeric if column not in binary]),
            ("binary", SimpleImputer(strategy="most_frequent"), binary),
            ("category", make_pipeline(
                SimpleImputer(strategy="constant", fill_value="(missing)"),
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ), categorical),
        ],
        sparse_threshold=0,
    )

def binary_columns(features, numeric):
    return [
        column for column in numeric
        if features[column].dropna().isin([0, 1]).all()
    ]

def local_neighbors(features):
    finder = NearestNeighbors(
        n_neighbors=NEIGHBORS - 1, metric="euclidean", algorithm="brute", n_jobs=1,
    ).fit(features)
    other_neighbors = finder.kneighbors(return_distance=False)
    return np.column_stack([np.arange(len(features)), other_neighbors])

def fairness(outcome, group, neighbors):
    rate_0 = outcome[group == 0].mean()
    rate_1 = outcome[group == 1].mean()
    consistency = 1 - np.mean(np.abs(outcome - outcome[neighbors].mean(axis=1)))
    return {
        "Rate group 0": rate_0,
        "Rate reference": rate_1,
        "DP_Diff": rate_0 - rate_1,
        "DIR": rate_0 / rate_1 if rate_1 else np.nan,
        "Consistency": consistency,
    }

records = []
for name, (features, label, group) in datasets.items():
    full_features = make_encoder(features).fit_transform(features)
    records.append({
        "Dataset": name, "Stage": "All original labels", "n": len(label),
        **fairness(label, group, local_neighbors(full_features)),
    })
display(pd.DataFrame(records).round(4))

,Dataset,Stage,n,Rate group 0,Rate reference,DP_Diff,DIR,Consistency
0,German Credit,All original labels,1000,0.5789,0.7284,-0.1494,0.7948,0.7404
1,Income survey,All original labels,8993,0.2395,0.2488,-0.0093,0.9626,0.7921
2,Titanic,All original labels,891,0.1889,0.7420,-0.5531,0.2546,0.7100


## Train, reweigh and evaluate

Stratify an 80:20 split jointly by group and label. Fit identical L2
logistic regressions with `C=1`, `lbfgs`, 2,000 maximum iterations,
no class weights and class prediction at the default 0.5 threshold. The
reweighed model uses the closed-form training weights
$w(a,y)=n_a n_y/(n\,n_{a,y})$ directly in the loss. The original paper
used the same weights for resampling with replacement. Direct weighting
represents the intended weighted distribution exactly and eliminates the
resampling variation. The protected attribute is used only for stratification,
reweighing and auditing.

Use the same test neighborhood graph for the original test labels and both
prediction vectors. Compare the two model rows to estimate the effect of
reweighing for this split, and use the test-label row to put both models in context.

In [7]:
def reweigh(label, group):
    counts = np.zeros((2, 2), dtype=int)
    np.add.at(counts, (group, label), 1)
    if np.any(counts == 0):
        raise ValueError("Each training group must contain both labels.")
    expected = np.outer(counts.sum(axis=1), counts.sum(axis=0)) / counts.sum()
    weights_by_cell = expected / counts
    weights = weights_by_cell[group, label]

    # Verify the weighted training joint counts factorize as intended.
    weighted_counts = np.zeros_like(expected)
    np.add.at(weighted_counts, (group, label), weights)
    np.testing.assert_allclose(weighted_counts, expected)
    return weights, weights_by_cell

def evaluate(name, features, label, group):
    train, test = train_test_split(
        np.arange(len(label)), test_size=TEST_SIZE,
        random_state=SEED, stratify=2 * group + label,
    )
    x_train, x_test = features.iloc[train], features.iloc[test]
    y_train, y_test = label[train], label[test]
    a_train, a_test = group[train], group[test]

    encoder = make_encoder(x_train)
    x_train = encoder.fit_transform(x_train)
    x_test = encoder.transform(x_test)
    neighbors = local_neighbors(x_test)
    weights, factors = reweigh(y_train, a_train)

    results = [{
        "Dataset": name, "Stage": "Test original labels", "n": len(test),
        **fairness(y_test, a_test, neighbors),
    }]
    for stage, sample_weight in [
        ("Test baseline", None), ("Test reweighed", weights),
    ]:
        classifier = LogisticRegression(
            C=1.0, solver="lbfgs", max_iter=2000, random_state=SEED,
        ).fit(x_train, y_train, sample_weight=sample_weight)
        predictions = classifier.predict(x_test)
        results.append({
            "Dataset": name, "Stage": stage, "n": len(test),
            **fairness(predictions, a_test, neighbors),
            "Accuracy": accuracy_score(y_test, predictions),
            "Balanced accuracy": balanced_accuracy_score(y_test, predictions),
        })
    settings = {
        "Dataset": name, "Train n": len(train), "Test group 0 n": int((a_test == 0).sum()),
        "Test ref. n": int((a_test == 1).sum()),
        "w(0,0)": factors[0, 0], "w(0,1)": factors[0, 1],
        "w(1,0)": factors[1, 0], "w(1,1)": factors[1, 1],
    }
    return results, settings

weight_rows = []
for name, (features, label, group) in datasets.items():
    rows, settings = evaluate(name, features, label, group)
    records.extend(rows)
    weight_rows.append(settings)

results = pd.DataFrame(records)
weights_table = pd.DataFrame(weight_rows)
display(results.sort_values("Dataset", kind="stable").round(4))
display(weights_table.round(4))

,Dataset,Stage,n,Rate group 0,Rate reference,DP_Diff,DIR,Consistency,Accuracy,Balanced accuracy
0,German Credit,All original labels,1000,0.5789,0.7284,-0.1494,0.7948,0.7404,NaN,NaN
3,German Credit,Test original labels,200,0.5789,0.7284,-0.1494,0.7948,0.7160,NaN,NaN
4,German Credit,Test baseline,200,0.6579,0.8580,-0.2001,0.7668,0.8490,0.7700,0.6690
5,German Credit,Test reweighed,200,0.7632,0.8519,-0.0887,0.8959,0.8600,0.7750,0.6679
1,Income survey,All original labels,8993,0.2395,0.2488,-0.0093,0.9626,0.7921,NaN,NaN
6,Income survey,Test original labels,1799,0.2398,0.2491,-0.0092,0.9629,0.7810,NaN,NaN
7,Income survey,Test baseline,1799,0.1697,0.1374,0.0323,1.2350,0.9389,0.8266,0.7033
8,Income survey,Test reweighed,1799,0.1697,0.1374,0.0323,1.2350,0.9389,0.8266,0.7033
2,Titanic,All original labels,891,0.1889,0.7420,-0.5531,0.2546,0.7100,NaN,NaN
9,Titanic,Test original labels,179,0.1897,0.7460,-0.5564,0.2542,0.7564,NaN,NaN


,Dataset,Train n,Test group 0 n,Test ref. n,"w(0,0)","w(0,1)","w(1,0)","w(1,1)"
0,German Credit,800,38,162,0.7125,1.2091,1.1045,0.9610
1,Income survey,7194,984,815,0.9944,1.0176,1.0068,0.9795
2,Titanic,712,116,63,0.7600,2.0317,2.3809,0.5174


In [8]:
before = results.query("Stage == 'Test baseline'").set_index("Dataset")
after = results.query("Stage == 'Test reweighed'").set_index("Dataset")
comparison = pd.DataFrame({
    "Change in absolute DP gap": after["DP_Diff"].abs() - before["DP_Diff"].abs(),
    "Change in DIR distance from 1": (after["DIR"] - 1).abs() - (before["DIR"] - 1).abs(),
    "Change in consistency": after["Consistency"] - before["Consistency"],
    "Change in accuracy": after["Accuracy"] - before["Accuracy"],
    "Change in balanced accuracy": after["Balanced accuracy"] - before["Balanced accuracy"],
})
display(comparison.round(4))

,Change in absolute DP gap,Change in DIR distance from 1,Change in consistency,Change in accuracy,Change in balanced accuracy
Dataset,,,,,
German Credit,-0.1114,-0.1291,0.0110,0.0050,-0.0012
Income survey,0.0000,0.0000,0.0000,0.0000,0.0000
Titanic,-0.0300,-0.1308,0.0346,0.0112,0.0280
